In [1]:
import os

# Aktuelles Arbeitsverzeichnis nehmen
script_dir = os.getcwd()

# Eine Ebene hoch
main_dir = os.path.abspath(os.path.join(script_dir, ".."))
os.chdir(main_dir)
import sys
if main_dir not in sys.path:
    sys.path.insert(0, main_dir)

print(f"📁 Arbeitsverzeichnis gesetzt auf: {os.getcwd()}")

📁 Arbeitsverzeichnis gesetzt auf: /Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies


In [2]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import re
import psycopg2
from functions import read_db_credentials, connect_to_db, load_json_data_from_db_as_json, save_df_to_db, data_from_data_sink

In [3]:
def save_df_to_db(df, table_name):
    creds = read_db_credentials()
    conn = connect_to_db(creds)
    cursor = conn.cursor()

    df = df.where(pd.notnull(df), None)  # NaN → None

    columns = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(df.columns))

    # dynamisch UPDATE-Anweisung generieren
    update_assignments = ', '.join([
        f"{col} = EXCLUDED.{col}" for col in df.columns if col != "user_number"
    ])

    insert_query = f"""
        INSERT INTO {table_name} ({columns})
        VALUES ({placeholders})
        ON CONFLICT (user_number) DO UPDATE
        SET {update_assignments}
    """

    for _, row in df.iterrows():
        cursor.execute(insert_query, row.tolist())

    conn.commit()
    cursor.close()
    conn.close()



In [4]:
with open("data/cur_user_selected.txt", "r", encoding="utf-8") as f:
    user = int(f.read().strip())

# with open("linkedin_profile Kopie.json") as f:
#     linkedin_data = json.load(f)
# def load_json_data_from_db_as_json(user, source ):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)

#     query = f"SELECT raw_json FROM fact_raw_data WHERE data_source = '{source}' AND user_number = {user} ;"
    
#     cursor = conn.cursor()
#     cursor.execute(query)
#     result = cursor.fetchone()
#     conn.close()
#     #return (result)
#     # # Parsen der JSON-Inhalte aus der 'data'-Spalte
#     parsed_data = result[0] if result else {}


#     # # Rückgabe als JSON-String (optional indent für Lesbarkeit)
#     return parsed_data


linkedin_data = load_json_data_from_db_as_json(user, "linkedin")
print(linkedin_data)

with open("data/test.json", 'w', encoding='utf-8') as f:
    json.dump(linkedin_data, f, ensure_ascii=False, indent=2)

{'SearchQueries': [{'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'ассистент-волонтёр'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'ассистент-волонтёр'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'волонтер социальной службы'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'волонтер социальной службы'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'волонтёр'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'волонтёр'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'студент-волонтёр'}, {'Time': '2023/07/03 21:59:59 UTC', 'Search Query': 'студент-волонтёр'}, {'Time': '2023/07/03 22:00:13 UTC', 'Search Query': 'volunteer OR volunteer staff OR student volunteer OR community volunteer OR volunteer assistant'}, {'Time': '2023/07/03 22:14:24 UTC', 'Search Query': 'volunteer OR volunteer staff OR student volunteer OR community volunteer OR volunteer assistant'}, {'Time': '2024/10/08 21:53:52 UTC', 'Search Query': 'Студент / практикант'}, {'Time':

In [5]:
# Top-Level Keys anzeigen
data = linkedin_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


"SearchQueries": 
  [
    "Time": str
    "Search Query": str
  ]
"Education": 
  [
    "School Name": str
    "Start Date": str
    "End Date": str
    "Notes": NoneType
    "Degree Name": str
    "Activities": NoneType
  ]
"Inferences_about_you": 
  [
    "Category": str
    "Type of inference": str
    "Description": str
    "Inference": str
  ]
"Member_Follows": 
  [
    "Date": str
    "Status": str
    "FullName": str
  ]
"messages": 
  [
    <empty>
  ]
"Job Seeker Preferences": 
  [
    "Locations": str
    "Industries": NoneType
    "Company Employee Count": NoneType
    "Preferred Job Types": str
    "Job Titles": str
    "Open To Recruiters": str
    "Dream Companies": NoneType
    "Profile Shared With Job Poster": NoneType
    "Job Title For Searching Fast Growing Companies": NoneType
    "Introduction Statement": NoneType
    "Phone Number": NoneType
    "Job Seeker Activity Level": NoneType
    "Preferred Start Time Range": NoneType
    "Commute Preference Starting Addres

In [6]:
def categorize_experience(years) -> str:
    """
    Categorizes work experience based on total years.

    Categories:
    - Junior: < 2 years
    - Senior: 2-6 years
    - Expert: > 6 years

    :param profile: Dictionary with key "total_experience_years"
    :return: Category as a string
    """
    
    if years < 2:
        return "Junior"
    elif 3 <= years <= 6:
        return "Senior"
    else:
        return "Expert"


In [7]:
print(linkedin_data.keys())

dict_keys(['SearchQueries', 'Education', 'Inferences_about_you', 'Member_Follows', 'messages', 'Job Seeker Preferences', 'Skills', 'SavedJobAlerts', 'Registration', 'Invitations', 'Logins', 'LearningCoachMessages', 'Profile Summary', 'learning_role_play_messages', 'Profile', 'Reactions', 'learning_coach_messages', 'Company Follows', 'Whatsapp Phone Numbers', 'coach_messages', 'LAN Ads Engagement', 'Rich_Media', 'PhoneNumbers', 'Ad_Targeting', 'Ads Clicked', 'Email Addresses'])


# Experience

In [10]:
positions = linkedin_data.get("Positions")
print(positions)
if positions == None:
    exp_years = 0
else:
    # Letzter Eintrag (angenommen: chronologisch sortiert)
    last_position = positions[-1]
    print(last_position)
    # Hole das Startdatum
    start_str = last_position.get("Started On")  # z. B. "Mar 2020"
    print(start_str)
    # Extrahiere das Jahr
    start_year = int(start_str.split()[-1])  # "2020" → 2020
    print(start_year)
    # Heutiges Jahr
    current_year = datetime.now().year

    # Differenz berechnen
    exp_years = current_year - start_year
print(exp_years)
exp_type = categorize_experience(exp_years)
print(exp_type)


None
0
Junior


# SKILLS

In [11]:


# Regex vorbereiten: Wortgrenzen, escape für Sonderzeichen
def to_regex(skill):
    return r'\b' + re.escape(skill) + r'\b'
# Beispielhafte Bewertungsskala:
# 5 = sehr wertvoll (Schlüsseltechnologien, gefragte Skills)
# 4 = hoch (oft gefordert, zentrale Tools)
# 3 = mittel (relevant, aber eher unterstützend)
# 2 = gering (spezifisch oder ergänzend)
# 1 = nice to have (nischig oder optional)

# Bewertungsliste (vereinfachte Einteilung nach Bedeutung/Verbreitung)
high_value = {
    4: ["Python", "SQL", "Machine Learning", "AWS", "Azure", "JavaScript", "React", "Docker", "Git",  "Communication", "Project Management"],
    3: ["Tableau", "Power BI", "Natural Language Processing","TensorFlow", "NLP","Pandas", "NumPy", "Java", "Node.js", "GitHub", "Jenkins", "Excel","Kubernetes", "Google Cloud Platform", "Agile", "Scrum", "Leadership", "Data Analysis", "Problem Solving", "HTML", "CSS", "Data Engineering", "Django", "C#", "MySQL", "PostgreSQL", "MongoDB"],
}

management_keywords = [
        "Project Management", "Product Management", "Stakeholder Management", "Leadership", "Scrum", "Agile",
        "Kanban", "Risk Management", "Roadmapping", "OKRs", "KPIs", "Jira", "Confluence", "Trello", "Asana",
        "SAFe", "Waterfall", "PRINCE2", "Lean", "Business Analysis", "Digital Transformation","BPMN","management"
    ]
it_keywords = [
            "Python", "R", "SQL", "Java", "JavaScript", "TypeScript", "HTML", "CSS", "PHP", "C++", "C#", ".NET",
            "Spring Boot", "Django", "Flask", "Ruby on Rails", "FastAPI", "Next.js", "Express.js", "GraphQL",
            "REST APIs", "Git", "GitHub", "GitLab", "Bitbucket", "Jest", "Cypress", "Selenium", "Playwright",
            "Webpack", "Babel", "NPM", "Yarn", "Linux", "Shell Scripting", "Bash", "Software Architecture",
            "Microservices", "AWS", "Azure", "Google Cloud Platform", "GCP", "Docker", "Kubernetes",
            "Terraform", "Ansible", "CI/CD", "Jenkins", "GitHub Actions", "CircleCI", "Travis CI", "CloudFormation",
            "Prometheus", "Grafana", "New Relic", "ELK Stack", "Logstash", "Kibana", "Helm", "Istio", "Serverless",
            "Vagrant", "MySQL", "PostgreSQL", "MongoDB", "Redis", "SQLite", "Oracle", "MariaDB", "Firebase","Data",
            "Cassandra", "DynamoDB", "Couchbase", "Elasticsearch", "InfluxDB", "Looker", "Snowflake", "Databricks",
            "BigQuery", "ETL", "Apache Spark", "Tableau", "Power BI", "Google Analytics", "Data Engineering",
            "Data Mining", "A/B Testing", "Scikit-learn", "TensorFlow", "PyTorch", "Keras", "Statistics", "Pandas",
            "NumPy", "Computer Vision", "Natural Language Processing", "NLP", "OCR", "Speech Recognition",
            "AR/VR", "Blockchain", "Web3", "Smart Contracts", "IoT", "Edge Computing", "Cybersecurity",
            "Penetration Testing", "Network Security", "Ethical Hacking", "SaaS", "PaaS", "IaaS", "API Design",
            "WebSockets", "OAuth2", "SAML", "Figma", "Sketch", "Adobe XD", "Adobe Photoshop", "Adobe Illustrator","VBA","Microsoft","Google Sheets", "Streamprocessing",
            "InVision"
        ]
communication_keywords = [
        "Communication", "Presentation Skills", "Negotiation", "Stakeholder Communication", "Teamwork", "Mentoring",
        "Empathy", "Conflict Resolution", "Emotional Intelligence", "English", "German", "Spanish", "French",
        "Hindi", "Mandarin", "Arabic", "Portuguese", "Russian","Marketing"
    ]
problem_solving_keywords = [
        "Problem Solving", "Decision Making", "Critical Thinking", "Adaptability", "Creativity",
        "Process Optimization", "Data Analysis", "Data Visualization", "Business Intelligence",
        "Requirement Analysis", "User Research", "Design Thinking", "Interaction Design", "Wireframing",
        "Prototyping", "Motion Design", "Typography", "Responsive Design", "Branding", "Accessibility","Analytische Fähigkeiten"
    ]
all_keywords = set(management_keywords + it_keywords + communication_keywords + problem_solving_keywords)

# 2. Alle Skills, die noch nicht in den Kategorien sind (also "sonstige")
extra_skills = ['Projektmanagement', 'Internet of Things (IoT)', 'Natural Language Processing (NLP)', 'Visual Basic for Applications (VBA)', 'Cascading Style Sheets (CSS)', 'Analytische Fähigkeiten', 'Microsoft Office', 'Vertrieb']
skills = sorted(list(all_keywords.union(extra_skills)))



def assign_category(skill):
    if skill in management_keywords:
        return "Management"
    elif skill in it_keywords:
        return "IT"
    elif skill in communication_keywords:
        return "Communication"
    elif skill in problem_solving_keywords:
        return "Problem Solving"
    else:
        return "Others"



# Skill-Rating berechnen
def rate_skill(skill):
    for rating, keywords in high_value.items():
        if skill in keywords:
            return rating
    return 2 if skill in df_skills["skill"].values else 1

# DataFrame erstellen
df_skills = pd.DataFrame({
    "skill": skills,
    "regex": [to_regex(skill) for skill in skills]
})
df_skills["rating"] = df_skills["skill"].apply(rate_skill)
df_skills["category"] = df_skills["skill"].apply(assign_category)



# Funktion zum Matchen von Skills anhand Regex mit Rückgabe von Rating und Kategorie
def match_skills_with_metadata(skills_to_match, skill_df):
    matched = []
    for input_skill in skills_to_match:
        found = False
        for _, row in skill_df.iterrows():
            if re.search(row["regex"], input_skill, re.IGNORECASE):
                matched.append({
                    "input": input_skill,
                    "matched_skill": row["skill"],
                    "rating": row["rating"],
                    "category": row["category"]
                })
                found = True
                break
        if not found:
            matched.append({
                "input": input_skill,
                "matched_skill": None,
                "rating": None,
                "category": "Not Found"
            })
    matched = pd.DataFrame(matched)
    matched["length_score"] = (matched["input"].str.len() - 13).abs()
    matched = matched.sort_values(by=["rating", "length_score"], ascending=[False, True]).reset_index()
    matched = matched.drop(columns=["length_score"])
    return matched

for i in range(0,len(df_skills)):
    print(df_skills.iloc[i])


skill            .NET
regex       \b\.NET\b
rating              2
category           IT
Name: 0, dtype: object
skill            A/B Testing
regex       \bA/B\ Testing\b
rating                     2
category                  IT
Name: 1, dtype: object
skill            API Design
regex       \bAPI\ Design\b
rating                    2
category                 IT
Name: 2, dtype: object
skill           AR/VR
regex       \bAR/VR\b
rating              2
category           IT
Name: 3, dtype: object
skill           AWS
regex       \bAWS\b
rating            4
category         IT
Name: 4, dtype: object
skill           Accessibility
regex       \bAccessibility\b
rating                      2
category      Problem Solving
Name: 5, dtype: object
skill           Adaptability
regex       \bAdaptability\b
rating                     2
category     Problem Solving
Name: 6, dtype: object
skill            Adobe Illustrator
regex       \bAdobe\ Illustrator\b
rating                           2
category      

In [12]:
skills = linkedin_data.get("Skills")
skills= [skill.get("Name") for skill in skills]
print(skills)
if not skills:  # None, [] oder andere leere Werte
    skills = ["not maintained", "not maintained", "not maintained"]
print(skills)
print(df_skills)
skills_processed = match_skills_with_metadata(skills, df_skills)
print(skills_processed)
top_skill_1 = skills_processed["input"][0]
top_skill_2 = skills_processed["input"][1]
top_skill_3 = skills_processed["input"][2]

category_counts = skills_processed["category"].value_counts(normalize=True) * 100

# In DataFrame umwandeln
df_category_distribution = category_counts.reset_index()
df_category_distribution.columns = ["category", "relative_frequency_percent"]

skill_cat_1 = f'{df_category_distribution["category"][0]} ({df_category_distribution["relative_frequency_percent"][0]}%)'
if len(df_category_distribution) > 1:
    skill_cat_2 = f'{df_category_distribution["category"][1]} ({df_category_distribution["relative_frequency_percent"][1]}%)'
else:
    skill_cat_2 = "NA"

['Машинное обучение', 'Большие массивы данных', 'Наука о данных', 'BPMN', 'Моделирование процессов', 'Управление проектами', 'бизнес анализ', 'Анализ данных', 'Системный анализ', 'исследование рынка ит', 'Реляционные базы данных', 'Базы данных', 'Экономическая оценка', 'ИТ-консалтинг', 'Моделирование ИС']
['Машинное обучение', 'Большие массивы данных', 'Наука о данных', 'BPMN', 'Моделирование процессов', 'Управление проектами', 'бизнес анализ', 'Анализ данных', 'Системный анализ', 'исследование рынка ит', 'Реляционные базы данных', 'Базы данных', 'Экономическая оценка', 'ИТ-консалтинг', 'Моделирование ИС']
           skill             regex  rating         category
0           .NET         \b\.NET\b       2               IT
1    A/B Testing  \bA/B\ Testing\b       2               IT
2     API Design   \bAPI\ Design\b       2               IT
3          AR/VR         \bAR/VR\b       2               IT
4            AWS           \bAWS\b       4               IT
..           ...          

# SKILL DIVERSITY INDEX

In [13]:
valid_categories = skills_processed['category'].dropna().unique()
num_categories = len(valid_categories)

# Score berechnen
score = num_categories / 5  # Es gibt 5 mögliche Kategorien laut Diagramm

# Klassifikation anhand des Scores
if score > 0.6:
    classification = "Well Rounded"
elif score > 0.4:
    classification = "Specialised"
else:
    classification = "Focused"

skill_diversity_index = score
skill_diversity_cat = classification

In [21]:
print(positions)
if positions == None:
    last_position = "not found"
else:
    positions = linkedin_data.get("Positions")
    print(positions)
    # Letzter Eintrag (angenommen: chronologisch sortiert)
    last_position = positions[0]
    print(last_position.get("Finished On"))
    if (last_position.get("Finished On") == None):
        date = f'since {last_position.get("Started On") }'
    else:
        date = f'from  {last_position.get("Started On")} until {last_position.get("Finished On")}'

    last_position = f'{last_position.get("Title")} at {last_position.get("Company Name")} \n in {last_position.get("Location")} \n {date}' 
    last_position

    

print(last_position)

None
not found


In [22]:

activity_times = data_from_data_sink(f"WITH id AS ( SELECT DISTINCT at_id FROM dim_health WHERE user_number = {user}) SELECT * FROM dim_activity_times WHERE at_id IN (SELECT at_id FROM id);")


/Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies/functions.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [23]:
import pandas as pd

def classify_working_type(df: pd.DataFrame) -> str:
    """
    Klassifiziert das Aktivitätsmuster einer Person basierend auf dem Zeitanteil
    früher, später und Wochenendaktivität.

    Voraussetzungen:
    - df enthält Spalten: 'weekday' (0=Montag, ..., 6=Sonntag), 'day_part' (Early, Late, Typical)

    Returns:
    - Kategorie als String: Early Bird, Midnight, Weekend, Typical
    """
    total = len(df)
    if total == 0:
        return "No Data"

    total = len(df)

    # Definierte Tageszeiten-Kategorien
    early_parts = ["Early Morning\n(4–8)", "Morning\n(8–12) "]
    late_parts = ["Evening\n(20–24)", "Night\n(0–4)"]
    weekend = ["Saturday", "Sunday"]

    early_pct = df["day_part"].isin(early_parts).sum() / total * 100
    late_pct = df["day_part"].isin(late_parts).sum() / total * 100
    weekend_pct = df["weekday"].isin(weekend ).sum() / total * 100

    if early_pct > 40:
        return "Early Bird"
    elif late_pct > 30:
        return "Midnight"
    elif weekend_pct > 50:
        return "Weekend"
    else:
        return "Typical"


working_type = classify_working_type(activity_times)

In [24]:

res = pd.DataFrame([{
    "user_number": user,
    "exp_years": exp_years,
    "exp_type": exp_type,
    "top_skill_1": top_skill_1,
    "top_skill_2": top_skill_2,
    "top_skill_3": top_skill_3,
    "skill_div_cat": skill_diversity_cat,
    "skill_div_index": skill_diversity_index,
    "skill_cat_1": skill_cat_1,
    "skill_cat_2": skill_cat_2 ,
    "cur_job": last_position,
    "working_type": working_type
}])
print(res)


save_df_to_db(res, "dim_linkedin")

   user_number  exp_years exp_type top_skill_1    top_skill_2    top_skill_3  \
0            1          0   Junior        BPMN  бизнес анализ  Анализ данных   

  skill_div_cat  skill_div_index                     skill_cat_1  \
0       Focused              0.4  Not Found (93.33333333333333%)   

                       skill_cat_2    cur_job working_type  
0  Management (6.666666666666667%)  not found      No Data  
